[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r1/r1-q3/01_classifier.ipynb)

# R1-Q3 Week 2 — Train a stress classifier

### This notebook produces a trained classifier, a test split, and an accuracy report.

This notebook trains a multi-class stress classifier on AtGenExpress so that you have a working model whose predictions you can later attribute. The classifier itself is the input to the SHAP attribution analysis in `02_shap.ipynb`, and its headline performance numbers will go into your Week 2 EQ#2 report.

By the end of this notebook you will have:

- A trained multi-class stress classifier saved to your output directory (`classifier.pkl`), ready for SHAP attribution in Week 3.
- Held-out test-set performance metrics (overall accuracy, per-class precision and recall) saved as `classifier_metrics.parquet`.
- A `classifier_summary.json` recording the model architecture, training hyperparameters, train/test split sizes, and headline performance, ready as Week 2 EQ#2 input.

In [ ]:
# Cell 0 — probe
try:
    import irilab2026 as iri
    print(f"irilab2026 {iri.__version__} already installed.")
    print("If you want to pull the latest changes, run Cell 1 below.")
except ModuleNotFoundError:
    print("irilab2026 not installed — run Cell 1 below.")

In [ ]:
# Cell 1 — install
# First run in a fresh Colab session: run this cell.
# Later runs in the same session: skip this cell.
!pip install git+https://github.com/geraldmc/irilab2026.git@main

**A note about a possible restart prompt.** If Colab shows a "RESTART SESSION" banner after the install: click it, wait for the kernel to reconnect, then continue from the next cell. The install does not need to be run again. If no banner appears, ignore this and move on.

In [ ]:
# Cell 2 — mount and setup
# Mount Drive, set up irilab2026, point to the R1-Q3 output directory.
import irilab2026 as iri
iri.setup()

OUTPUT_DIR = iri.output_dir("r1_q3")
print(f"Output directory: {OUTPUT_DIR}")

### 1) Load your precommitments from Notebook 00

You wrote four decisions to `precommit.json` at the end of Notebook 00. This section reads them back, checks them, and makes the ones Notebook 01 needs available as Python variables.

Two reasons to read defensively rather than trust the file:

- **A typo upstream is cheap to catch here and expensive to catch later.** If `data_source_and_scope.source` is misspelled, Section 2's loader will branch the wrong way and you won't notice until Section 4 fails with a confusing error.
- **Notebooks 02 and 03 inherit the same file.** Validating all four top-level keys here — not just the one Notebook 01 actually uses — means a malformed precommit fails immediately rather than surfacing as a SHAP error in Week 3 or a gene-overlap error in Week 4.

What gets checked:

- All four top-level keys are present: `attribution_method`, `reference_gene_set`, `artifact_like_definition`, `data_source_and_scope`.
- `attribution_method.method` is one of `"SHAP"`, `"integrated_gradients"`, `"permutation_importance"`.
- `data_source_and_scope.source` is one of `"GEO"`, `"TAIR_NASCArrays"`.
- `data_source_and_scope.n_stress_classes` matches the source (8 for GEO, 9 for TAIR/NASCArrays).
- `artifact_like_definition` carries its six expected fields (`n_classes`, `per_class`, `overlap_clause`, `metadata_clause`, `combination`, `rollup`) plus `rationale`.
- **Cross-field consistency:** `artifact_like_definition.n_classes` equals `data_source_and_scope.n_stress_classes`. If they disagree, the class count you committed to in Precommit 3 doesn't match what Precommit 4 says you'll actually have data for. Go back to Notebook 00 Cell 5.D or Cell 6.D and reconcile before continuing.
- `reference_gene_set.primary_source` is non-empty.

A `ValueError` from the cell below means your precommit is malformed; the message will tell you which key. Fix it in Notebook 00 and re-run that notebook's Section 7 to regenerate the file before continuing here.

In [ ]:
import json

# Read the precommit file written by Notebook 00.
precommit_path = OUTPUT_DIR / "precommit.json"
if not precommit_path.exists():
    raise FileNotFoundError(
        f"precommit.json not found at {precommit_path}. "
        "Run Notebook 00 to completion (including Section 7) before continuing."
    )

with open(precommit_path, "r") as f:
    precommit = json.load(f)

# --- Top-level structure ---

expected_keys = {
    "attribution_method",
    "reference_gene_set",
    "artifact_like_definition",
    "data_source_and_scope",
}
missing = expected_keys - set(precommit.keys())
if missing:
    raise ValueError(
        f"precommit.json is missing top-level keys: {sorted(missing)}. "
        "Re-run Notebook 00 Sections 3-6 to regenerate."
    )

# --- attribution_method ---

valid_methods = {"SHAP", "integrated_gradients", "permutation_importance"}
method = precommit["attribution_method"].get("method")
if method not in valid_methods:
    raise ValueError(
        f"attribution_method.method is '{method}'; "
        f"expected one of {sorted(valid_methods)}. "
        "Edit Notebook 00 Cell 3.C and re-run Section 7."
    )

# --- data_source_and_scope ---

if precommit["data_source_and_scope"].get("source") != "GEO":
    raise ValueError(
        f"data_source_and_scope.source is "
        f"{precommit['data_source_and_scope'].get('source')!r}; "
        "expected 'GEO' (the only path the irilab2026 library currently serves). "
        "Edit Notebook 00 Cell 6.D and re-run Section 7."
    )

n_stress_classes = precommit["data_source_and_scope"].get("n_stress_classes")
if n_stress_classes != 8:
    raise ValueError(
        f"data_source_and_scope.n_stress_classes is {n_stress_classes}; "
        "expected 8 for the GEO range. "
        "Edit Notebook 00 Cell 6.D and re-run Section 7."
    )

# --- artifact_like_definition ---

expected_alf_keys = {
    "n_classes", "per_class", "overlap_clause", "metadata_clause",
    "combination", "rollup", "rationale",
}
alf = precommit["artifact_like_definition"]
missing_alf = expected_alf_keys - set(alf.keys())
if missing_alf:
    raise ValueError(
        f"artifact_like_definition is missing fields: {sorted(missing_alf)}. "
        "Re-run Notebook 00 Section 5."
    )

# Cross-field consistency: n_classes must match between Precommit 3 and Precommit 4.
if alf["n_classes"] != n_stress_classes:
    raise ValueError(
        f"artifact_like_definition.n_classes ({alf['n_classes']}) "
        f"does not match data_source_and_scope.n_stress_classes ({n_stress_classes}). "
        "Reconcile Notebook 00 Cell 5.D and Cell 6.D before continuing."
    )

# --- reference_gene_set ---

primary_source = precommit["reference_gene_set"].get("primary_source", "")
if not primary_source.strip():
    raise ValueError(
        "reference_gene_set.primary_source is empty. "
        "Edit Notebook 00 Cell 4.C and re-run Section 7."
    )

# --- Unpack for downstream sections ---

n_classes = n_stress_classes    # used throughout

# --- Summary ---

print("precommit.json loaded and validated.")
print(f"  Data source:        GEO")
print(f"  Stress classes:     {n_classes}")
print(f"  Attribution method: {method}")
print(f"  Reference set:      {primary_source}")
print(f"  Artifact-like rule: {alf['combination']}-combined across {n_classes} classes")

`precommit.json` is on disk, structurally sound, and unpacked into two variables — `data_source` and `n_classes` — that the rest of this notebook will use. The full `precommit` dict is also in scope; downstream sections reach into it directly when they need specific fields.

What each top-level key is for downstream:

- `data_source_and_scope` — **Section 2 reads this.** Branches the AtGenExpress loader based on `data_source`.
- `attribution_method` — validated here, used in **Notebook 02** to pick the attribution code path.
- `reference_gene_set` — validated here, used in **Notebook 03** to build the gene-set overlap test.
- `artifact_like_definition` — validated here, used in **Notebook 03** to apply the artifact-like rule per class. The cross-field check between `n_classes` and `n_stress_classes` was asserted in Cell 1.C so a mismatch surfaces at the top of Week 2 rather than at the bottom of Week 4.

Notebook 01 itself only branches on `data_source`. The other three keys ride along untouched.

### 2) Load AtGenExpress

You committed in Notebook 00 Section 6 to one of two AtGenExpress download paths — the GEO range (8 stresses, no oxidative) or the full TAIR/NASCArrays release (9 stresses). This section calls the matching loader, runs structural checks on what comes back, and prints the class × tissue distribution table.

The table is the headline output. Section 3's tissue-handling decision (per-tissue classifiers vs. tissue-as-covariate) hinges on what this table looks like — whether shoot and root samples are roughly balanced across stress classes, whether any class is shoot-only or root-only, and whether the imbalance is severe enough to force one path.

What this section does *not* do:

- Re-walk the AtGenExpress experimental design. Notebook 00 Section 2 covered that.
- Re-flag the GSE-batch / stress-class confound. Notebook 00 Section 2 covered that too; Precommit 3's metadata clause is the formal response. This section just notices the confound is still present in the data you're about to train on, so it remains the thing the artifact-like rule guards against.

What gets sanity-checked:

- Shape and sample-ID alignment between the expression matrix and the metadata DataFrame.
- Required metadata columns are present: `stress_class`, `tissue`, `timepoint`, `gse_accession`.
- Number of distinct non-control stress classes matches `n_classes`.
- Both `shoot` and `root` tissue values are present (the cross-tabulation needs both to make sense in Section 3).

A failure here usually means the loader returned something unexpected — most often a version mismatch with `irilab2026`. The error message will name what to check.

In [ ]:
# Load AtGenExpress GEO. The library splits this into two calls — load_atgenexpress
# for the per-stress expression DataFrames, and atgenexpress_metadata for the sample
# annotations. The two calls share a cache (and load_atgenexpress's SOFT-file
# downloads feed atgenexpress_metadata's parsing), so the second call is cheap once
# the first has run.
import pandas as pd

per_stress = iri.load_atgenexpress()      # dict[stress_name, DataFrame(probes x samples)]
meta_raw = iri.atgenexpress_metadata()    # DataFrame indexed by GSM

# Assemble a single probes x samples matrix by concatenating the per-stress
# DataFrames along the sample axis. All DataFrames share the ATH1 probe row index,
# so concat aligns rows automatically and produces one wide matrix with every
# GSM as a column.
expr = pd.concat(per_stress.values(), axis=1)

# Rename library columns to the names the rest of this notebook uses. The library
# helper uses 'stress', 'time_h', 'gse'; Sections 3-6 below use 'stress_class',
# 'timepoint', 'gse_accession' (which read better in a classifier context). The
# translation lives here so both name choices can stand.
meta = meta_raw.rename(columns={
    "stress": "stress_class",
    "time_h": "timepoint",
    "gse": "gse_accession",
})

# --- Structural checks ---

# expr is probes x samples; meta is samples (GSM) x annotations.
if expr.shape[1] != meta.shape[0]:
    raise ValueError(
        f"Sample count mismatch: expr has {expr.shape[1]} columns, "
        f"meta has {meta.shape[0]} rows. The two loader calls returned "
        "different sets of GSMs."
    )

if set(expr.columns) != set(meta.index):
    extra_in_expr = sorted(set(expr.columns) - set(meta.index))
    extra_in_meta = sorted(set(meta.index) - set(expr.columns))
    raise ValueError(
        f"Sample IDs don't align between expr and meta. "
        f"In expr but not meta ({len(extra_in_expr)}): {extra_in_expr[:5]}... "
        f"In meta but not expr ({len(extra_in_meta)}): {extra_in_meta[:5]}..."
    )

# Reorder meta to match expr's column order. Section 4's split indexes both
# objects by position, so a mismatch here would silently scramble labels.
meta = meta.loc[expr.columns]

required_cols = {"stress_class", "tissue", "timepoint", "gse_accession", "last_update_date"}
missing_cols = required_cols - set(meta.columns)
if missing_cols:
    raise ValueError(
        f"meta is missing expected columns: {sorted(missing_cols)}. "
        "This usually means the irilab2026 library is older than the release "
        "that added last_update_date. Run "
        "`pip install --upgrade git+https://github.com/geraldmc/irilab2026.git@main` "
        "and restart the runtime."
    )

# --- Class count check (non-control stress classes) ---

stress_classes_present = sorted(
    meta.loc[meta["stress_class"] != "control", "stress_class"].unique()
)
if len(stress_classes_present) != n_classes:
    raise ValueError(
        f"Found {len(stress_classes_present)} distinct non-control stress classes "
        f"({stress_classes_present}); precommit says n_classes = {n_classes}. "
        "Verify your Precommit 4 choice matches what the loader actually returns."
    )

# --- Tissue check ---

tissues_present = sorted(meta["tissue"].unique())
if not {"shoot", "root"}.issubset(set(tissues_present)):
    raise ValueError(
        f"Expected both 'shoot' and 'root' in meta['tissue']; found {tissues_present}."
    )

# --- Summary and the headline table ---

print("Loaded AtGenExpress (GEO range).")
print(f"  Expression matrix:  {expr.shape[0]:,} probes x {expr.shape[1]:,} samples")
print(f"  Stress classes:     {len(stress_classes_present)} non-control "
      f"({stress_classes_present})")
print( "  Plus control samples (shown in the table below).")
print(f"  Tissues:            {tissues_present}")
print()
print("Class x tissue distribution:")
class_tissue = pd.crosstab(meta["stress_class"], meta["tissue"], margins=True)
display(class_tissue)

Three things to look at in the table above:

- **Per-class balance.** For each stress class, how do shoot and root counts compare? A roughly even split means the per-tissue classifier path in Section 3 has enough samples on either side. A heavy lean one way means one of the per-tissue classifiers will be undertrained for that class.
- **Tissue-only classes.** Any stress row where shoot or root is zero? AtGenExpress profiled most stresses in both tissues, but check — if any stress is single-tissue, the per-tissue path can't cover it without dropping that class from one of the two classifiers.
- **Class imbalance.** Look at the `All` column. If one class has 3-4× more samples than another, your classifier will pay more attention to the over-represented class regardless of tissue handling. Stratified splitting in Section 4 partly addresses this; severe imbalance may also be worth a note in your EQ#2 report.

The `control` row is in the table because the loader includes control samples — Section 4 will decide what to do with them (include as a class, hold out as background, or drop). The non-control row count above is what matches `n_classes`.

**The GSE-batch confound is still here.** Notebook 00 Section 2 made the point that stress class and GSE accession are nearly collinear in AtGenExpress — most classes were profiled in a single GSE. That hasn't changed; the loader just packaged what's on disk. Your Precommit 3 metadata clause (the variance-partitioning test against GSE accession, processing date, and tissue identity) is what guards against the model exploiting that collinearity. Nothing to do about it here; just keeping it in view.

**Forward to Section 3.** The class × tissue table above is the primary input to Section 3's decision. Re-read the page Considerations bullet 1 before continuing — Section 3 commits the answer.

### 3) Tissue handling

AtGenExpress profiles both shoot and root tissue across most stresses. The R1-Q3 page Considerations bullet 1 names two defensible ways to handle this:

**Option A — Per-tissue classifiers.** Train two separate classifiers, one on shoot samples only and one on root samples only. Each operates within a single tissue's expression baseline, so tissue identity can't act as a hidden shortcut — the model never sees both tissues at once, so it has nothing to latch onto.

- Pros: methodologically clean; the "did the model learn biology or tissue?" question becomes "did it learn biology within each tissue?", which is sharper.
- Costs: half the sample count per classifier; classes with skewed tissue distribution (look at the table in Section 2) may be under-supported on one side. Notebook 02 runs SHAP twice and Notebook 03 produces two artifact-like verdicts.

**Option B — Tissue-as-covariate.** Train one classifier on pooled data, with tissue added explicitly as a feature column. The model can use tissue, but only by attributing to it openly — SHAP will surface tissue's contribution alongside genes.

- Pros: full sample size for every class; one classifier, one SHAP analysis, one artifact-like verdict.
- Costs: tissue can still act as a near-shortcut if its effect is large relative to stress-class effects. Your Precommit 3 metadata clause already partitions variance against tissue identity, so the artifact-like rule will catch a tissue-dominated model — but you should still expect tissue to show up as a top SHAP feature for at least some classes.

The page's framing — "Either split your training by tissue or model tissue explicitly so it can't act as a hidden shortcut" — names both as acceptable. The choice depends on what the Section 2 class × tissue table looks like for your data source (severe imbalance pushes toward Option B; clean balance makes Option A workable) and on how much methodological cleanness you want at the cost of statistical power.

**This decision is yours.** It isn't in `precommit.json` because it's a Week 2 modeling choice, not a Week 1 commitment. But it gets recorded the same way precommit decisions did — to `classifier_summary.json`, which Notebook 02 reads to know which classifier path to attribute over.

In [ ]:
# Initialize the running summary dict for this notebook's modeling decisions.
# Sections 4 and 5 will append to it; Section 6 writes it to classifier_summary.json.
classifier_summary = {}

# Record your tissue handling decision.
# Fill in both fields; leaving them as None or "..." will raise below.
classifier_summary["tissue_handling"] = {
    "choice": ...,       # "per_tissue" or "tissue_as_covariate"
    "rationale": ...,    # 1-3 sentences explaining why this option fits your data and aims
}

# --- Validation ---

valid_choices = {"per_tissue", "tissue_as_covariate"}
choice = classifier_summary["tissue_handling"]["choice"]
if choice not in valid_choices:
    raise ValueError(
        f"tissue_handling.choice is {choice!r}; "
        f"expected one of {sorted(valid_choices)}."
    )

rationale = classifier_summary["tissue_handling"]["rationale"]
if not isinstance(rationale, str) or not rationale.strip() or rationale.strip() == "...":
    raise ValueError(
        "tissue_handling.rationale is empty or a placeholder. "
        "Write 1-3 sentences explaining your choice before continuing."
    )

print(f"Recorded tissue handling: {choice}")
print(f"  Rationale: {rationale}")

### Practice 3.1 — Apply your tissue handling decision

In Cell 3.C above, you recorded `classifier_summary["tissue_handling"]["choice"]` as either `"per_tissue"` or `"tissue_as_covariate"`. In the code cell below, build a dict called `tissue_subsets` that captures the consequences of that choice for Section 4's training step.

The dict has a fixed shape that Section 4 expects:

- **If your choice is `"per_tissue"`**, `tissue_subsets` has two entries:
  - `"shoot"`: a tuple `(expr_shoot, meta_shoot)` containing only shoot samples.
  - `"root"`: a tuple `(expr_root, meta_root)` containing only root samples.
- **If your choice is `"tissue_as_covariate"`**, `tissue_subsets` has one entry:
  - `"pooled"`: a tuple `(expr, meta)` — the full dataset, with tissue still as a column in `meta`. Section 4 will add tissue as a feature at training time.

The scaffold below has the `if`/`elif`/`else` set up for you. Replace each `raise NotImplementedError` with the code that builds the right `tissue_subsets` for that branch. Section 2's `data_source` branching showed the same pattern applied to a different decision.

In [ ]:
choice = classifier_summary["tissue_handling"]["choice"]

if choice == "per_tissue":
    # Practice 3.1 — per_tissue branch
    # Build tissue_subsets as a dict with two entries: "shoot" and "root".
    # Each value is a tuple (expr_subset, meta_subset) filtered to that tissue.
    raise NotImplementedError("Practice 3.1 — per_tissue branch: see markdown above")

elif choice == "tissue_as_covariate":
    # Practice 3.1 — tissue_as_covariate branch
    # Build tissue_subsets as a dict with one entry: "pooled".
    # The value is the tuple (expr, meta) — no filtering.
    raise NotImplementedError("Practice 3.1 — tissue_as_covariate branch: see markdown above")

else:
    raise ValueError(
        f"Unknown tissue_handling.choice: {choice!r}. "
        "Edit Cell 3.C and use 'per_tissue' or 'tissue_as_covariate'."
    )

# Quick structural check — runs after the student's fill.
assert isinstance(tissue_subsets, dict), "tissue_subsets must be a dict"
for key, value in tissue_subsets.items():
    assert isinstance(value, tuple) and len(value) == 2, (
        f"tissue_subsets[{key!r}] must be a (expr, meta) tuple"
    )
    expr_sub, meta_sub = value
    assert expr_sub.shape[1] == meta_sub.shape[0], (
        f"tissue_subsets[{key!r}]: sample count mismatch between expr and meta"
    )

print(f"Built tissue_subsets with {len(tissue_subsets)} entry/entries: "
      f"{sorted(tissue_subsets.keys())}")
for key, (expr_sub, _) in tissue_subsets.items():
    print(f"  {key}: {expr_sub.shape[1]} samples")

You now have:

- `classifier_summary` — a dict holding your `tissue_handling` block. Sections 4 and 5 will append to it (`train_test_split`, `training`, `accuracy_gate`); Section 6 writes it to `classifier_summary.json` for Notebook 02 to read.
- `tissue_subsets` — a dict that Section 4 iterates over to produce the train/test split and train the classifier(s). One entry for `"tissue_as_covariate"`, two entries for `"per_tissue"`.

Section 4 doesn't branch on your choice — it just iterates `tissue_subsets`. That keeps the rest of the notebook the same shape regardless of which path you took here. Notebook 02 reads `classifier_summary["tissue_handling"]["choice"]` to know whether to load one classifier or two.

### 4) Train/test split, train, evaluate

#### 4.1 Train/test split

Two things happen in this subsection.

**Hold controls out for Notebook 02.** Your `tissue_subsets` dict currently contains all samples — stress samples plus controls. Controls aren't training data here; they're the reference distribution Notebook 02's SHAP analysis will use to compute attributions ("how does this stress sample differ from a typical control?"). Section 4.1's first step separates controls out and stores them in a `controls_set` object that mirrors the shape of `tissue_subsets`. Section 6 will save it to `controls_background.parquet` for Notebook 02 to read.

**Stratified split on the stress samples.** What's left in each tissue subset is stress samples only. The split is stratified by `stress_class` so each class is represented in both train and test sets in roughly its original proportion. Without stratification, a minority class (look at the `All` column of Section 2's table) can end up entirely in train or entirely in test, which makes evaluation meaningless.

Mentor-given defaults — recorded for reproducibility, not student-chosen:

- **Test fraction: 0.25.** Standard sklearn default; gives ~25% test, ~75% train. The choice between 0.20 and 0.30 is a real one for small per-class samples, but it isn't where the methodological action is in this notebook. Recorded, not interrogated.
- **Random state: 42.** A fixed seed so re-running the notebook gives the same split. The accuracy gate in Section 5 will lose its meaning if the test set changes between runs.

What gets recorded to `classifier_summary["train_test_split"]`:

- `strategy: "stratified"`
- `test_fraction: 0.25`
- `random_state: 42`
- `stratify_by: "stress_class"`
- `controls_held_out: true`
- `n_controls`, plus per-subset `n_train`, `n_test`, and per-class counts in train/test.

These numbers go into Notebook 01's EQ#2 report alongside the accuracy verdict.

In [ ]:
from sklearn.model_selection import train_test_split

# --- Separate controls from stress samples ---

# controls_set mirrors the shape of tissue_subsets:
#   - per_tissue: {"shoot": (expr_ctl_shoot, meta_ctl_shoot), "root": (expr_ctl_root, meta_ctl_root)}
#   - tissue_as_covariate: {"pooled": (expr_ctl_pooled, meta_ctl_pooled)}
# stress_subsets has the same shape, with controls removed.
controls_set = {}
stress_subsets = {}

for key, (expr_sub, meta_sub) in tissue_subsets.items():
    is_control = meta_sub["stress_class"] == "control"
    control_ids = meta_sub.index[is_control]
    stress_ids = meta_sub.index[~is_control]

    controls_set[key] = (expr_sub.loc[:, control_ids], meta_sub.loc[control_ids])
    stress_subsets[key] = (expr_sub.loc[:, stress_ids], meta_sub.loc[stress_ids])

n_controls_total = sum(meta_ctl.shape[0] for _, meta_ctl in controls_set.values())
print(f"Held out {n_controls_total} control samples for Notebook 02's SHAP background.")
for key, (_, meta_ctl) in controls_set.items():
    print(f"  {key}: {meta_ctl.shape[0]} controls")

# --- Initialize the train_test_split block ---

# Mentor-given parameters. Recorded here so they appear in classifier_summary.json
# and the EQ#2 report. Not student-chosen.
TEST_FRACTION = 0.25
RANDOM_STATE = 42

classifier_summary["train_test_split"] = {
    "strategy": "stratified",
    "test_fraction": TEST_FRACTION,
    "random_state": RANDOM_STATE,
    "stratify_by": "stress_class",
    "controls_held_out": True,
    "n_controls": n_controls_total,
    "per_subset": {},  # filled in by the practice cell below
}

### Practice 4.1.1 — Stratified split on each tissue subset

In the code cell below, walk `stress_subsets` and produce a `splits` dict that Section 4.2's training loop will iterate over. The expected shape is:

```python
splits = {
    "shoot": {
        "X_train": ...,  # genes x samples_train (DataFrame, expression matrix subset)
        "X_test":  ...,  # genes x samples_test
        "y_train": ...,  # length samples_train (Series, stress_class labels)
        "y_test":  ...,  # length samples_test
    },
    "root": { ... },     # same shape (only present if tissue_handling.choice == "per_tissue")
}
```

Use `sklearn.model_selection.train_test_split` with `stratify=meta_sub["stress_class"]`, `test_size=TEST_FRACTION`, and `random_state=RANDOM_STATE` (the constants defined in Cell 4.1.C).

**Shape note.** sklearn's `train_test_split` expects `X` as samples × features (rows = samples). The expression matrix `expr_sub` from `stress_subsets` is genes × samples (rows = genes). You'll need to transpose before passing it in. The `splits` dict above keeps `X_train` and `X_test` in samples × features orientation — sklearn's classifier in Section 4.2 expects the same. Don't transpose back.

Two things to keep an eye on as you fill this in:

- **Per-class minimum.** Stratified splitting requires at least 2 samples per class (one for train, one for test). If `stress_subsets["shoot"]` has only 1 sample for some class after the control hold-out, `train_test_split` will raise. The fix is upstream — either drop that class from the per-tissue path, or switch to tissue-as-covariate in Section 3 and re-run. The page Considerations bullet 1 anticipated this; the Section 2 class × tissue table is where you'd have seen it coming.
- **Per-subset counts get recorded.** After building each subset's split, append a record to `classifier_summary["train_test_split"]["per_subset"][key]` containing `n_train`, `n_test`, `train_class_counts` (a dict from class label to count), and `test_class_counts`. The skeleton below shows the structure.

In [ ]:
splits = {}

for key, (expr_sub, meta_sub) in stress_subsets.items():
    # X: samples x features (transpose expr_sub from genes x samples).
    # y: stress_class labels aligned to X's row order.
    X_full = expr_sub.T
    y_full = meta_sub["stress_class"]

    # Practice 4.1.1 — fill the stratified split call.
    # Use train_test_split with stratify=y_full, test_size=TEST_FRACTION,
    # random_state=RANDOM_STATE.
    X_train, X_test, y_train, y_test = ...

    splits[key] = {
        "X_train": X_train,
        "X_test":  X_test,
        "y_train": y_train,
        "y_test":  y_test,
    }

    # Record per-subset counts. This block runs after your fill above.
    classifier_summary["train_test_split"]["per_subset"][key] = {
        "n_train": int(X_train.shape[0]),
        "n_test":  int(X_test.shape[0]),
        "train_class_counts": y_train.value_counts().to_dict(),
        "test_class_counts":  y_test.value_counts().to_dict(),
    }

    print(f"Split {key}: {X_train.shape[0]} train, {X_test.shape[0]} test "
          f"({X_train.shape[1]} features)")

You now have three objects ready for Section 4.2:

- `splits` — a dict (one or two entries) holding `X_train`, `X_test`, `y_train`, `y_test` per tissue subset. Section 4.2's training loop iterates this.
- `controls_set` — held out from training, saved to disk by Section 6, read by Notebook 02 as the SHAP background.
- `classifier_summary["train_test_split"]` — the parameter record, including per-class counts in train and test for each subset. Eyeball this dict before continuing; the per-class counts are your first look at how stratification distributed each class.

If you went per-tissue and any per-class count in train is below ~3, the classifier in Section 4.2 will be undertrained for that class regardless of accuracy on other classes. That's a structural limitation of the data, not a bug — but it's the kind of thing to mention in your EQ#2 report.

#### 4.2 Train

You have a `splits` dict from Section 4.1. This subsection fits a `RandomForestClassifier` per subset and stores the fitted models in a `classifiers` dict that Section 4.3 evaluates and Section 6 saves to `classifier.pkl`.

**Why Random Forest.** Three reasons. First, it handles wide data (thousands of features, dozens to hundreds of samples) without preprocessing — no need to scale, center, or select genes upfront. Second, it's robust to class imbalance when given `class_weight="balanced"`, which matters for AtGenExpress's uneven class counts (you saw those in Section 2's table). Third, it's tree-based, so Notebook 02's SHAP attribution uses `TreeExplainer` — an exact, fast attribution method rather than the slow sampling-based explainers other model families need.

**Tissue-as-covariate handling.** If you chose `"tissue_as_covariate"` in Section 3, this cell adds tissue as an explicit feature column to every train, test, and controls matrix before fitting. The page Considerations bullet 1 named this as the whole point of Option B — tissue is available to the model, but only as a feature it has to attribute to openly, so Notebook 02's SHAP analysis surfaces tissue's contribution alongside gene contributions. If you chose `"per_tissue"`, this block is a no-op (the conditional skips it; each classifier already operates within a single tissue).

**Mentor-given hyperparameters.** These are recorded to `classifier_summary["training"]["hyperparameters"]` so they appear in the EQ#2 report and the saved JSON.

- `n_estimators=200` — more trees averages out the noise floor; default 100 is on the low side for ~22K features.
- `max_features="sqrt"` — sklearn default; ~148 features considered per split. Random feature subsetting per split is what makes a forest a forest rather than 200 correlated trees.
- `class_weight="balanced"` — re-weights samples inversely to class frequency. Without this, minority stress classes (look at your Section 2 counts) get under-fit.
- `random_state=42` — matches Section 4.1's `RANDOM_STATE` so re-running the notebook gives the same trees on the same split. Section 5's accuracy gate loses its meaning if the model changes between runs.
- `n_jobs=-1` — uses all available CPU cores. Colab's CPU is modest; this gives you the speed-up that exists.

What gets recorded to `classifier_summary["training"]`:

- `classifier_type` — the sklearn class name, cross-checked against the pickle in Notebook 02.
- `hyperparameters` — every parameter listed above, for reproducibility.
- `per_subset[key]` — `n_train_samples`, `n_features` (gene count plus tissue columns if added), `fit_seconds`. The fit time isn't methodologically important but is a useful gut-check that the loop did what you expected.


In [ ]:
import time
from sklearn.ensemble import RandomForestClassifier

# Mentor-given hyperparameters. Recorded to classifier_summary so they appear
# in the EQ#2 report and the saved JSON.
N_ESTIMATORS    = 200
MAX_FEATURES    = "sqrt"
CLASS_WEIGHT    = "balanced"
RF_RANDOM_STATE = 42         # matches Section 4.1's RANDOM_STATE
N_JOBS          = -1

# --- Tissue-as-covariate: add tissue feature columns ---
# If tissue-as-covariate, append one-hot tissue columns to every train, test,
# and controls matrix BEFORE fitting. Otherwise this block is skipped.

if classifier_summary["tissue_handling"]["choice"] == "tissue_as_covariate":
    _, meta_stress_pooled = stress_subsets["pooled"]
    _, meta_ctl_pooled    = controls_set["pooled"]

    # Build one tissue one-hot encoding spanning ALL sample_ids (stress + controls)
    # so the column set is consistent across train, test, and controls matrices.
    all_meta = pd.concat([meta_stress_pooled, meta_ctl_pooled])
    tissue_onehot  = pd.get_dummies(all_meta["tissue"], prefix="tissue").astype(int)
    tissue_columns = list(tissue_onehot.columns)
    print(f"Added tissue one-hot columns: {tissue_columns}")

    # Append to train and test (already samples x features).
    splits["pooled"]["X_train"] = splits["pooled"]["X_train"].join(tissue_onehot)
    splits["pooled"]["X_test"]  = splits["pooled"]["X_test"].join(tissue_onehot)

    # Append to controls. controls_set stores expr as genes x samples, so
    # transpose, join the per-sample tissue columns, transpose back.
    ctl_expr, ctl_meta = controls_set["pooled"]
    ctl_expr_with_tissue = ctl_expr.T.join(tissue_onehot).T
    controls_set["pooled"] = (ctl_expr_with_tissue, ctl_meta)
    print(f"Tissue columns appended to controls_set['pooled'].")
else:
    print("Per-tissue mode: no tissue feature columns added (each classifier "
          "operates within a single tissue).")

# --- Train one RandomForestClassifier per subset ---

classifiers = {}
classifier_summary["training"] = {
    "classifier_type": "RandomForestClassifier",
    "hyperparameters": {
        "n_estimators":  N_ESTIMATORS,
        "max_features":  MAX_FEATURES,
        "class_weight":  CLASS_WEIGHT,
        "random_state":  RF_RANDOM_STATE,
        "n_jobs":        N_JOBS,
    },
    "per_subset": {},
}

print()
print("Fitting classifiers:")
for key, split in splits.items():
    t0 = time.time()
    clf = RandomForestClassifier(
        n_estimators  = N_ESTIMATORS,
        max_features  = MAX_FEATURES,
        class_weight  = CLASS_WEIGHT,
        random_state  = RF_RANDOM_STATE,
        n_jobs        = N_JOBS,
    ).fit(split["X_train"], split["y_train"])
    fit_seconds = time.time() - t0

    classifiers[key] = clf
    classifier_summary["training"]["per_subset"][key] = {
        "n_train_samples": int(split["X_train"].shape[0]),
        "n_features":      int(split["X_train"].shape[1]),
        "fit_seconds":     round(fit_seconds, 2),
    }
    print(f"  {key}: {split['X_train'].shape[0]} samples x "
          f"{split['X_train'].shape[1]} features  ->  fit in {fit_seconds:.1f}s")


You now have:

- `classifiers` — a dict keyed by subset, each value a fitted `RandomForestClassifier`. Section 4.3 calls `.predict` on each; Section 6 pickles the whole dict.
- `classifier_summary["training"]` — the hyperparameters and per-subset fit stats. These flow into `classifier_summary.json` and the EQ#2 report.

Section 4.3 evaluates these classifiers on the held-out test sets.


#### 4.3 Evaluate

This subsection calls `.predict` on each classifier's test set, computes performance metrics, and records them to `classifier_summary["evaluation"]["per_subset"][key]`. Section 5's accuracy gate reads `balanced_accuracy` from each entry to decide whether the classifier is worth attributing.

**Why balanced accuracy as the gate metric.** AtGenExpress class counts are uneven — you saw this in Section 2's table. Raw accuracy weights large classes more heavily, so a model that ignores minority classes can post a high raw-accuracy number while being useless for the stresses with the fewest samples. Balanced accuracy averages per-class recall, which is closer to "did the model learn each stress class". Section 5's threshold rationale assumes this metric.

**What gets recorded per subset:**

- `balanced_accuracy` — the Section 5 contract. Section 5 raises if this key is missing.
- `raw_accuracy` — for context in the EQ#2 report.
- `per_class_recall` / `per_class_precision` — dicts mapping each class label to its score. The minority classes are where the headline number can mislead.
- `confusion_matrix` — nested list with `class_labels` recorded alongside so row/column order is unambiguous.
- `class_labels` — sorted list of class names. Confusion matrix rows and columns are in this order.

This is also where you'd see the classic Random-Forest-on-genomics pathology, if it happens: high raw accuracy, balanced accuracy that lags by 10-20 points, and most of the gap concentrated in one or two minority stress classes. The per-class recall dict is where to look.


In [ ]:
from sklearn.metrics import (
    balanced_accuracy_score,
    accuracy_score,
    classification_report,
    confusion_matrix,
)

classifier_summary["evaluation"] = {"per_subset": {}}

print("Evaluating classifiers on held-out test sets:")
for key, clf in classifiers.items():
    split  = splits[key]
    y_true = split["y_test"]
    y_pred = clf.predict(split["X_test"])

    bal_acc = balanced_accuracy_score(y_true, y_pred)
    raw_acc = accuracy_score(y_true, y_pred)

    # Per-class metrics. Pin the label order so dict iteration and the
    # confusion matrix rows/cols stay aligned.
    class_labels = sorted(y_true.unique())
    report = classification_report(
        y_true, y_pred,
        labels        = class_labels,
        output_dict   = True,
        zero_division = 0,
    )
    per_class_recall    = {c: float(report[c]["recall"])    for c in class_labels}
    per_class_precision = {c: float(report[c]["precision"]) for c in class_labels}

    cm = confusion_matrix(y_true, y_pred, labels=class_labels)

    classifier_summary["evaluation"]["per_subset"][key] = {
        "balanced_accuracy":   float(bal_acc),   # Section 5 reads this key.
        "raw_accuracy":        float(raw_acc),
        "per_class_recall":    per_class_recall,
        "per_class_precision": per_class_precision,
        "confusion_matrix":    cm.tolist(),
        "class_labels":        class_labels,
    }

    print(f"  {key}:")
    print(f"    balanced_accuracy = {bal_acc:.3f}")
    print(f"    raw_accuracy      = {raw_acc:.3f}")
    print(f"    n_classes         = {len(class_labels)}  ({class_labels})")


`classifier_summary["evaluation"]["per_subset"]` now has one entry per fitted classifier, each with the balanced accuracy that Section 5's gate reads plus the ancillary metrics the EQ#2 report needs.

The headline numbers above are the first answer to the EQ#2 question. Before running Section 5, look at them: if balanced accuracy is at or near chance (1/8 = 0.125 for the GEO path, 1/9 = 0.111 for the TAIR/NASCArrays path), the model didn't learn — and no threshold choice in Section 5 will save it. The fix is upstream, not downstream.


### 5) Accuracy gate: is this worth attributing? (the explicit "stop here if not" check)

The R1-Q3 page Considerations bullet 2 frames this section directly:

> AtGenExpress is clean and well-separated, so a classifier can hit high test accuracy while learning features that have nothing to do with stress biology — accuracy alone tells you the model is consistent, not that it's right. Attribution on a poorly-performing classifier is wasted work; attribution on a model you haven't checked is worse, because you'll generate a story about features that don't actually drive predictions. Confirm classifier performance first, then attribute.

This is the cell where "confirm performance first" gets enforced. If your classifier doesn't meet the threshold, this section raises a `RuntimeError`, refuses to save the gate verdict to disk, and blocks Notebook 02 from running. That's by design — Notebook 02 spends real compute time on SHAP, and the output is only meaningful if the model is worth attributing.

**The threshold is yours to choose.** This is the one student-facing decision in Section 5. It isn't in `precommit.json` because the right floor depends on your class count (8 or 9) and what your training table looked like.

A defensible range for the kind of multi-class classifier this notebook trains:

- **Chance level.** 1/8 = 12.5% for the GEO path; 1/9 = 11.1% for the TAIR/NASCArrays path. Anything at or near chance means the model learned nothing.
- **Floor for "worth interpreting."** Roughly 50-65% balanced accuracy is where the range of defensible choices sits. Below ~50%, SHAP attributions will be noisy because the model is barely better than chance for several classes. Above ~65%, you're being more conservative — fine, but understand you may block yourself from interpreting a model that has real signal in 6 of 8 classes just because 2 are weak.
- **Above 75%.** Plausible for tissue-as-covariate on the GEO path, where pooled samples give the model more to work with. Less plausible for per-tissue paths where each classifier has half the samples. Don't anchor on a number you've seen in a Random Forest paper without checking what makes sense for your data.

**Metric.** Balanced accuracy, not raw accuracy. AtGenExpress class counts are uneven (you saw this in Section 2's table); raw accuracy weights large classes more heavily. Balanced accuracy averages per-class recall, which is closer to what "the classifier learned each stress" actually means.

**Pre-commit before you see the number.** Cell 5.C asks you to write down the threshold and a short rationale. Cell 5.D then surfaces the observed accuracy. If you change the threshold in Cell 5.C after seeing Cell 5.D's output, you're fitting your gate to your result — and the gate stops being a test. The notebook can't prevent you from doing this; the cell ordering is the prompt to do it honestly.

**What if you have multiple subsets?** Per-tissue mode produces two classifiers (shoot and root); each gets evaluated against the threshold independently. The overall verdict is computed mechanically: pass if both subsets pass, fail if both fail, partial otherwise. A partial result doesn't block Notebook 02 — but only the passing subsets get attributed.

In [ ]:
# Commit your threshold and rationale BEFORE running Cell 5.D.
# Cell 5.D surfaces the observed balanced accuracy; if you change the values
# below after seeing it, the gate stops being a real test.

threshold = ...              # a float between 0 and 1 (e.g., 0.60 for 60% balanced accuracy)
threshold_rationale = ...    # 1-3 sentences: why this floor for your class count and tissue handling

# --- Validation ---

if not isinstance(threshold, (int, float)):
    raise ValueError(
        f"threshold must be a number; got {type(threshold).__name__}."
    )
if not (0.0 < threshold < 1.0):
    raise ValueError(
        f"threshold is {threshold}; expected a value strictly between 0 and 1. "
        "(0.60 means 60% balanced accuracy.)"
    )

# Chance-level guard. Picking a threshold at or below chance defeats the gate.
chance = 1.0 / n_classes
if threshold <= chance:
    raise ValueError(
        f"threshold ({threshold:.3f}) is at or below chance ({chance:.3f}) "
        f"for an {n_classes}-class problem. Re-read Cell 5.B and pick a floor "
        "above chance level."
    )

if not isinstance(threshold_rationale, str) or not threshold_rationale.strip() or threshold_rationale.strip() == "...":
    raise ValueError(
        "threshold_rationale is empty or a placeholder. "
        "Write 1-3 sentences explaining your floor before continuing to Cell 5.D."
    )

# Initialize the accuracy_gate block. Cell 5.D and 5.E will append to it.
classifier_summary["accuracy_gate"] = {
    "metric": "balanced_accuracy",
    "threshold": float(threshold),
    "threshold_rationale": threshold_rationale.strip(),
}

print(f"Threshold committed: {threshold:.3f} balanced accuracy.")
print(f"Rationale: {threshold_rationale.strip()}")
print()
print("Run Cell 5.D to see your observed accuracy.")

In [ ]:
# Pull observed balanced accuracy from Section 4.3's evaluation block.
# If this KeyError fires, Section 4.3 didn't run or didn't write the expected key.
if "evaluation" not in classifier_summary:
    raise RuntimeError(
        "classifier_summary['evaluation'] not found. "
        "Run Section 4.3 to completion before this cell."
    )

threshold = classifier_summary["accuracy_gate"]["threshold"]
per_subset_verdicts = {}

print(f"Threshold: {threshold:.3f} balanced accuracy.")
print(f"Observed per subset:")
print()

for key in classifier_summary["evaluation"]["per_subset"]:
    observed = classifier_summary["evaluation"]["per_subset"][key]["balanced_accuracy"]
    verdict = "pass" if observed >= threshold else "fail"
    per_subset_verdicts[key] = {
        "observed": float(observed),
        "verdict": verdict,
    }
    marker = "✓" if verdict == "pass" else "✗"
    print(f"  {marker} {key}:  observed = {observed:.3f}   verdict = {verdict}")

# Store mechanical verdicts for Cell 5.E to roll up.
classifier_summary["accuracy_gate"]["per_subset"] = per_subset_verdicts

print()
print("Run Cell 5.E to commit your interpretive rationale and finalize the gate.")

In [ ]:
# Write a short interpretation of the per-subset verdicts above.
# This is what reviewers (and your future self) will read in the EQ#2 report.

overall_rationale = ...    # 1-3 sentences: what do the per-subset verdicts mean for the analysis?

# --- Validation ---

if not isinstance(overall_rationale, str) or not overall_rationale.strip() or overall_rationale.strip() == "...":
    raise ValueError(
        "overall_rationale is empty or a placeholder. "
        "Write 1-3 sentences interpreting the per-subset verdicts before continuing."
    )

# --- Mechanical overall verdict ---

per_subset = classifier_summary["accuracy_gate"]["per_subset"]
verdicts = [v["verdict"] for v in per_subset.values()]

if all(v == "pass" for v in verdicts):
    overall_verdict = "pass"
elif all(v == "fail" for v in verdicts):
    overall_verdict = "fail"
else:
    overall_verdict = "partial"

classifier_summary["accuracy_gate"]["overall_verdict"] = overall_verdict
classifier_summary["accuracy_gate"]["overall_rationale"] = overall_rationale.strip()

# --- Hard stop on fail ---

if overall_verdict == "fail":
    # Refuse to proceed. Section 6's write to disk will not run.
    # Notebook 02 reads classifier_summary.json and will not have an accuracy_gate
    # block, which blocks the SHAP path.
    raise RuntimeError(
        f"Accuracy gate FAILED. Every subset's observed balanced accuracy "
        f"is below the threshold ({classifier_summary['accuracy_gate']['threshold']:.3f}). "
        "Attribution is not worth running. Options: revisit your tissue handling "
        "choice in Section 3, check for class-imbalance issues in Section 2's table, "
        "or accept that this data source and class scope do not support an interpretable "
        "classifier with the chosen method. Do NOT lower the threshold to make the gate pass."
    )

# Partial or pass: report and continue.
print(f"Overall verdict: {overall_verdict.upper()}")
print(f"Rationale: {overall_rationale.strip()}")
print()
if overall_verdict == "partial":
    passing = [k for k, v in per_subset.items() if v["verdict"] == "pass"]
    failing = [k for k, v in per_subset.items() if v["verdict"] == "fail"]
    print(f"Notebook 02 will attribute the passing subsets only: {passing}")
    print(f"Failing subsets blocked from attribution: {failing}")
else:
    print("All subsets pass. Notebook 02 will attribute every subset.")

The gate verdict is recorded in `classifier_summary["accuracy_gate"]`. Three downstream consumers:

- **Section 6** writes the dict to `classifier_summary.json`. If the gate failed, Cell 5.E raised before reaching here, so no JSON gets written — Notebook 02 reads `classifier_summary.json` at startup and will not find an `accuracy_gate` block, which is its signal to refuse to run.
- **Notebook 02** reads `accuracy_gate.overall_verdict` and `accuracy_gate.per_subset`. On a `partial` verdict, it attributes only the subsets marked `pass`. On a `pass` verdict, it attributes every subset.
- **EQ#2 report** (Week 2 deliverable). Your threshold, rationale, per-subset observed accuracies, and overall rationale all go into the report. Section 6 will assemble the numbers; the writing is yours.

What this section does not do: it does not catch the case where a classifier passes the threshold but learned the wrong thing. That's the artifact-like rule's job, which runs in Notebook 03. The accuracy gate just says: this is worth interpreting at all. Whether the interpretation reveals biology or technical confounds is the Week 4 question.

### 6) Closeout: save the trained model, the test split, and the accuracy report; submit EQ#2

This section saves four files to `OUTPUT_DIR` and reminds you what goes into the EQ#2 submission.

The four files are the contract with Notebook 02. That notebook reads all four at startup and refuses to run if any are missing or if the accuracy gate failed (in which case `classifier_summary.json` will not exist, by design — see Section 5).

- **`classifier.pkl`** — your fitted classifier or classifiers, in a dict keyed by tissue subset. One classifier for tissue-as-covariate, two for per-tissue.
- **`test_splits.parquet`** — the held-out test set for each subset, with sample IDs, the `stress_class` label, and all gene-expression columns. Notebook 02 runs SHAP on this.
- **`controls_background.parquet`** — the control samples held out in Section 4.1, with the same column shape as `test_splits.parquet`. Notebook 02 uses these as the SHAP reference distribution.
- **`classifier_summary.json`** — the metadata dict you built up across Sections 3, 4, and 5. Notebook 02 reads `accuracy_gate.per_subset` to decide which subsets to attribute. Notebook 03 reads `artifact_like_definition` (carried from `precommit.json`) and `tissue_handling.choice` (set in Section 3).

What's not saved:

- **Training set.** The random state plus the loader plus `precommit.json` lets anyone regenerate the exact train/test split. Saving the training set would double the disk footprint for no operational benefit.
- **Full expression matrix.** Notebook 02 reads expression from the test splits, which already have the feature columns. The full matrix lives at the loader level.
- **EQ#2 report.** That's a Week 2 paper-style deliverable you write externally. The last cell of this section prints a checklist of what to include.

A note on `.parquet`. Parquet is a column-oriented binary format that handles wide DataFrames (tens of thousands of gene columns) efficiently. pandas reads and writes it natively. If you've never used it before, the read syntax is the same as CSV: `pd.read_parquet(path)`.

In [ ]:
import pickle
from datetime import datetime, timezone

# --- Append closeout metadata to classifier_summary ---
# This block is the fourth top-level key; the first three (tissue_handling,
# train_test_split / training / evaluation, accuracy_gate) were filled by
# Sections 3-5.

classifier_summary["metadata"] = {
    "notebook_completed_at": datetime.now(timezone.utc).isoformat(),
    "irilab2026_version": iri.__version__,
    "data_source": data_source,
    "n_classes": n_classes,
    "classifier_type": "RandomForestClassifier",
}

# --- Write classifier.pkl ---

classifier_path = OUTPUT_DIR / "classifier.pkl"
with open(classifier_path, "wb") as f:
    pickle.dump(classifiers, f)

# Sanity check: file exists, non-trivial size.
size_kb = classifier_path.stat().st_size / 1024
if size_kb < 1:
    raise RuntimeError(
        f"classifier.pkl is suspiciously small ({size_kb:.1f} KB). "
        "Check that the classifiers dict was populated in Section 4.2."
    )
print(f"Wrote {classifier_path.name} ({size_kb:,.1f} KB).")
print(f"  Contains classifiers for: {sorted(classifiers.keys())}")

In [ ]:
# Assemble test splits into a single wide DataFrame.
# One row per test sample; columns: subset, sample_id, stress_class, then features.

test_frames = []
for key, split in splits.items():
    df = split["X_test"].copy()
    df.insert(0, "stress_class", split["y_test"].values)
    df.insert(0, "subset", key)
    df.index.name = "sample_id"
    test_frames.append(df.reset_index())

test_splits_df = pd.concat(test_frames, ignore_index=True)

test_splits_path = OUTPUT_DIR / "test_splits.parquet"
test_splits_df.to_parquet(test_splits_path, index=False)

size_mb = test_splits_path.stat().st_size / (1024 * 1024)
print(f"Wrote {test_splits_path.name} ({size_mb:,.2f} MB).")
print(f"  Shape: {test_splits_df.shape[0]:,} test samples x {test_splits_df.shape[1]:,} columns")
print(f"  Subsets: {sorted(test_splits_df['subset'].unique())}")

In [ ]:
# Assemble controls into the same wide shape as test_splits.parquet.
# One row per control sample; columns: subset, sample_id, stress_class, then features.

control_frames = []
for key, (expr_ctl, meta_ctl) in controls_set.items():
    # expr_ctl is genes x samples; transpose to samples x features.
    df = expr_ctl.T.copy()
    df.insert(0, "stress_class", meta_ctl["stress_class"].values)
    df.insert(0, "subset", key)
    df.index.name = "sample_id"
    control_frames.append(df.reset_index())

controls_df = pd.concat(control_frames, ignore_index=True)

controls_path = OUTPUT_DIR / "controls_background.parquet"
controls_df.to_parquet(controls_path, index=False)

size_mb = controls_path.stat().st_size / (1024 * 1024)
print(f"Wrote {controls_path.name} ({size_mb:,.2f} MB).")
print(f"  Shape: {controls_df.shape[0]:,} control samples x {controls_df.shape[1]:,} columns")
print(f"  Subsets: {sorted(controls_df['subset'].unique())}")

In [ ]:
# Write classifier_summary.json.

summary_path = OUTPUT_DIR / "classifier_summary.json"

# default=str handles datetime, numpy scalars, and a few other edge cases that
# can sneak into the dict from sklearn (e.g., np.int64 in class_counts).
with open(summary_path, "w") as f:
    json.dump(classifier_summary, f, indent=2, default=str)

# --- Read-back verification ---

with open(summary_path, "r") as f:
    summary_check = json.load(f)

expected_top_keys = {
    "tissue_handling", "train_test_split", "training",
    "evaluation", "accuracy_gate", "metadata",
}
missing = expected_top_keys - set(summary_check.keys())
if missing:
    raise RuntimeError(
        f"classifier_summary.json round-trip missing keys: {sorted(missing)}. "
        "Check that Sections 3, 4, and 5 all completed."
    )

if summary_check["accuracy_gate"]["overall_verdict"] == "fail":
    # Section 5 should have raised before reaching here; this branch is a safety net.
    raise RuntimeError(
        "classifier_summary.json shows a FAILED accuracy gate. "
        "Section 5 should have raised before writing this file."
    )

size_kb = summary_path.stat().st_size / 1024
print(f"Wrote {summary_path.name} ({size_kb:,.1f} KB).")
print(f"  Top-level keys: {sorted(summary_check.keys())}")
print(f"  Accuracy gate verdict: {summary_check['accuracy_gate']['overall_verdict']}")
print()
print("All four artifacts written to OUTPUT_DIR:")
for fname in ["classifier.pkl", "test_splits.parquet",
              "controls_background.parquet", "classifier_summary.json"]:
    fpath = OUTPUT_DIR / fname
    if fpath.exists():
        size = fpath.stat().st_size
        size_str = f"{size / (1024*1024):.2f} MB" if size > 1024*1024 else f"{size / 1024:.1f} KB"
        print(f"  ✓ {fname:<30s} ({size_str})")
    else:
        print(f"  ✗ {fname:<30s} (MISSING)")

### EQ#2 — Week 2 deliverable

EQ#2 is the Week 2 paper-style submission: a short writeup of your classifier and its accuracy verdict. You write it externally (Notion, doc, whatever the program is using); this section gives you the checklist and tells you where every number lives in `classifier_summary.json`.

What to cover:

- **Methodological choices.** Tissue handling (`classifier_summary["tissue_handling"]`), train/test split parameters (`classifier_summary["train_test_split"]`), classifier type and any hyperparameters (`classifier_summary["training"]`, `classifier_summary["metadata"]["classifier_type"]`), accuracy threshold and rationale (`classifier_summary["accuracy_gate"]["threshold"]`, `["threshold_rationale"]`).
- **The numbers.** Per-class counts in train and test for each subset (`classifier_summary["train_test_split"]["per_subset"]`), observed balanced accuracy per subset (`classifier_summary["evaluation"]["per_subset"][<subset>]["balanced_accuracy"]`), the threshold vs. each observed value, and the per-subset and overall verdicts (`classifier_summary["accuracy_gate"]["per_subset"]`, `["overall_verdict"]`).
- **The verdict and its interpretation.** Your overall rationale from Section 5 (`classifier_summary["accuracy_gate"]["overall_rationale"]`). For a partial verdict, name which subsets pass and what that means for Notebook 02's scope.
- **What's still open.** Anything you noticed in Section 2's class × tissue table that constrains the analysis — severe imbalance, single-tissue classes, the GSE-batch confound. The artifact-like rule in Notebook 03 will pick these up formally, but EQ#2 is where you flag them in writing.

What's not expected in EQ#2:

- SHAP attributions, gene-set overlap, the artifact-like verdict — those are Week 3 and Week 4 work. EQ#2 is a stopping point that says "the classifier is worth interpreting" (or "it isn't, and here's why").

When EQ#2 is submitted, you're done with Week 2. Notebook 02 picks up from `classifier_summary.json` and the three companion artifacts.